In [1]:
!pip install streamlit



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [59]:
%%writefile app.py
import io
import streamlit as st
import pandas as pd

from extraccion import (
    extraer_registros_patwin,
    extraer_registros_pdf,
    fusionar_registro_patwin_pdf,
)
from db import (
    init_db,
    bd_existe,
    DB_PATH,
    insertar_muestra_combinada,
)
from validacion_archivos import validar_excel_patwin, validar_pdf_mmt
from vista_historico import mostrar_paso_3



def init_session_state():
    if "step" not in st.session_state:
        st.session_state["step"] = 1
    if "excel_bytes" not in st.session_state:
        st.session_state["excel_bytes"] = None
    if "pdf_bytes" not in st.session_state:
        st.session_state["pdf_bytes"] = None
    if "ultimo_resumen" not in st.session_state:
        st.session_state["ultimo_resumen"] = None


def ir_a_paso(n):
    st.session_state["step"] = n


def main():
    init_session_state()

    st.title("TFG MammaTyper – Integración Patwin + MammaTyper")

    # ---- Sidebar: estado BD + navegación rápida ----
    st.sidebar.subheader("Base de datos")
    st.sidebar.write(f"📁 Archivo: `{DB_PATH}`")
    st.sidebar.write("Estado: siempre inicializada automáticamente")

    st.sidebar.markdown("---")
    st.sidebar.subheader("Navegación")
    if st.sidebar.button("Ir al paso 1"):
        ir_a_paso(1)
    if st.sidebar.button("Ir al paso 2"):
        ir_a_paso(2)
    if st.sidebar.button("Ir al paso 3"):
        ir_a_paso(3)

    step = st.session_state["step"]

    # =========================
    # PASO 1: subida de archivos
    # =========================
    if step == 1:
        st.header("Paso 1 · Subir archivos")

        st.markdown("Sube el **Excel** (Patwin/MammaTyper) y el **PDF** con los informes MammaTyper®.")

        col1, col2 = st.columns(2)
        with col1:
            archivo_excel = st.file_uploader("Excel (.xlsx)", type=["xlsx"], key="uploader_excel")
        with col2:
            archivo_pdf = st.file_uploader("PDF MammaTyper", type=["pdf"], key="uploader_pdf")

        if archivo_excel is not None:
            st.session_state["excel_bytes"] = archivo_excel.getvalue()
            st.success(f"Excel cargado: {archivo_excel.name}")

        if archivo_pdf is not None:
            st.session_state["pdf_bytes"] = archivo_pdf.getvalue()
            st.success(f"PDF cargado: {archivo_pdf.name}")

        st.markdown("---")
        if st.session_state["excel_bytes"] and st.session_state["pdf_bytes"]:
            st.info("Excel y PDF listos para procesar.")
            if st.button("Ir al paso 2 ▶️"):
                ir_a_paso(2)
        else:
            st.warning("Sube **ambos** archivos para continuar al paso 2.")

    # =========================
    # PASO 2: procesar y guardar
    # =========================
    elif step == 2:
        st.header("Paso 2 · Procesar y guardar en la base de datos")

        if not (st.session_state["excel_bytes"] and st.session_state["pdf_bytes"]):
            st.error("No hay archivos cargados. Vuelve al paso 1.")
            if st.button("⬅️ Volver al paso 1"):
                ir_a_paso(1)
            return

        st.markdown(
            "En este paso se valida el formato, se extraen los datos del Excel y el PDF, "
            "se fusionan por Sample ID y se guardan en la base de datos."
        )

        if st.button("Procesar y guardar muestras en BD"):

            # VALIDACIÓN DEL EXCEL Y PDF
            ok_xls, msg_xls = validar_excel_patwin(st.session_state["excel_bytes"])
            ok_pdf, msg_pdf = validar_pdf_mmt(st.session_state["pdf_bytes"])

            if not ok_xls:
                st.error(f"❌ Error en el Excel:\n\n{msg_xls}")
                return
            if not ok_pdf:
                st.error(f"❌ Error en el PDF:\n\n{msg_pdf}")
                return

            st.success("✔️ Excel y PDF con formato correcto. Procesando...")

            try:
                # 1) Extraer registros del Excel
                excel_bytes = st.session_state["excel_bytes"]
                excel_file = io.BytesIO(excel_bytes)
                registros_excel = extraer_registros_patwin(excel_file)

                # Índice por sample_id
                idx_excel = {
                    reg.get("sample_id"): reg
                    for reg in registros_excel
                    if reg.get("sample_id")
                }

                # 2) Extraer registros del PDF
                pdf_bytes = st.session_state["pdf_bytes"]
                pdf_file = io.BytesIO(pdf_bytes)
                regs_pdf = extraer_registros_pdf(pdf_file)

                procesados = []
                sin_match = []

                for reg_pdf in regs_pdf:
                    sid = reg_pdf.get("sample_id")
                    reg_excel = idx_excel.get(sid)

                    if reg_excel is None:
                        sin_match.append(sid)
                        continue

                    combinado = fusionar_registro_patwin_pdf(reg_excel, reg_pdf)
                    insertar_muestra_combinada(combinado)
                    procesados.append(combinado)

                st.session_state["ultimo_lote"] = procesados
                st.session_state["ultimo_resumen"] = {
                    "n_excel": len(registros_excel),
                    "n_pdf": len(regs_pdf),
                    "n_procesados": len(procesados),
                    "sin_match": sin_match,
                }

                st.success(f"Proceso finalizado. {len(procesados)} muestras guardadas.")

            except Exception as e:
                st.error(f"❌ Error procesando los archivos: {e}")

        st.markdown("---")

        col_prev, col_next = st.columns(2)
        with col_prev:
            if st.button("⬅️ Volver al paso 1"):
                ir_a_paso(1)
        with col_next:
            if st.button("Ir al paso 3 ▶️"):
                ir_a_paso(3)

    # =========================
    # PASO 3: histórico
    # =========================
    elif step == 3:
         mostrar_paso_3(ir_a_paso)


if __name__ == "__main__":
    main()


Overwriting app.py


In [54]:
import importlib, extraccion
importlib.reload(extraccion)

[f for f in dir(extraccion) if "extraer" in f or "fusionar" in f]


['_extraer_biomarcador',
 '_extraer_registro_pagina',
 'extraer_desde_pdf',
 'extraer_registros_patwin',
 'extraer_registros_pdf',
 'fusionar_registro_patwin_pdf']

In [53]:
%%writefile extraccion.py
import re
from typing import List, Dict, Any

import pandas as pd
from PyPDF2 import PdfReader


# =========================
#  EXCEL (Patwin / MMT)
# =========================

def ver_columnas_excel(file) -> list:
    df = pd.read_excel(file)
    return list(df.columns)


def _limpiar_valor(x: Any):
    if pd.isna(x):
        return None
    return x


def extraer_registros_patwin(file) -> List[Dict[str, Any]]:
    """
    Lee el Excel Patwin/MMT y devuelve una lista de registros (uno por fila válida).
    """
    df = pd.read_excel(file)

    if "Subtype Info MMT" in df.columns:
        mask_subtipo = ~df["Subtype Info MMT"].isna()
        df_subtipo = df[mask_subtipo].copy()
    else:
        df_subtipo = df.copy()

    registros: List[Dict[str, Any]] = []

    df_con_id = df_subtipo.copy()
    if "Sample ID" in df_con_id.columns:
        mask_id = ~df_con_id["Sample ID"].isna()
        df_con_id = df_con_id[mask_id]

    if not df_con_id.empty:
        iterable = df_con_id.iterrows()
        usar_demo_ids = False
    else:
        iterable = df_subtipo.iterrows()
        usar_demo_ids = True

    for idx, row in iterable:
        if usar_demo_ids:
            sample_id = f"DEMO_{idx}"
        else:
            sample_id = row.get("Sample ID")
            if pd.isna(sample_id):
                continue
            sample_id = str(sample_id)

        reg = {
            "nhc": None,
            "ronda": _limpiar_valor(row.get("RONDA")),
            "sample_id": sample_id,
            "celularidad": _limpiar_valor(row.get("% celularidad")),
            "subtipo_ihq": _limpiar_valor(row.get("Subtype APA IHQ")),
            "subtipo_mmt": _limpiar_valor(row.get("Subtype Info MMT")),
            "ERBB2_value": _limpiar_valor(row.get("ERBB2_value")),
            "ERBB2_status": _limpiar_valor(row.get("ERBB2_status")),
            "ERBB2_IHQ_SISH": _limpiar_valor(row.get("ERBB2 IHQ/SISH")),
            "ESR1_value": _limpiar_valor(row.get("ESR1_value")),
            "ESR1_status": _limpiar_valor(row.get("ESR1_status")),
            "ESR1_IHQ": _limpiar_valor(row.get("ESR1 IHQ")),
            "PGR_value": _limpiar_valor(row.get("PGR_value")),
            "PGR_status": _limpiar_valor(row.get("PGR_status")),
            "PGR_IHQ": _limpiar_valor(row.get("PGR-IHQ")),
            "MKI67_value": _limpiar_valor(row.get("MKI67_value")),
            "MKI67_status": _limpiar_valor(row.get("MKI67_status")),
            "KI67_IHQ": _limpiar_valor(row.get("KI67_IHQ")),
        }
        registros.append(reg)

    return registros


# =========================
#  PDF — varias páginas
# =========================

def _extraer_biomarcador(texto: str, nombre: str):
    patron = rf"{nombre}\s+([\d\.,]+)\s+(\w+)"
    m = re.search(patron, texto)
    if not m:
        return None, None
    valor_str = m.group(1).replace(",", ".")
    try:
        valor = float(valor_str)
    except ValueError:
        valor = None
    estado = m.group(2)
    return valor, estado


def _extraer_registro_pagina(texto: str) -> Dict[str, Any] | None:
    """Extrae UNA muestra desde UNA página del PDF."""
    lineas = [ln.strip() for ln in texto.splitlines() if ln.strip()]

    # Sample ID
    sample_id = None
    for ln in lineas:
        if "Sample ID:" in ln:
            raw = ln.split("Sample ID:")[1].strip()
            sample_id = raw.replace(" ", "")
            break

    if not sample_id:
        return None

    # Fecha
    fecha = None
    for ln in lineas:
        if "Date of report:" in ln:
            fecha = ln.split("Date of report:")[1].strip().split()[0]
            break

    # Subtipos
    subtipo_mmt = None
    subtipo_detalle = None

    idx = None
    for i, ln in enumerate(lineas):
        if "Subtype According" in ln:
            idx = i
            break

    if idx is not None:
        if idx + 2 < len(lineas):
            subtipo_mmt = lineas[idx + 2]
        if idx + 3 < len(lineas):
            subtipo_detalle = lineas[idx + 3]
            for cortar in ["Biomarker", "ResultsStatus"]:
                if cortar in subtipo_detalle:
                    subtipo_detalle = subtipo_detalle.split(cortar)[0].strip()

    erbb2_v, erbb2_s = _extraer_biomarcador(texto, "ERBB2")
    esr1_v, esr1_s = _extraer_biomarcador(texto, "ESR1")
    pgr_v, pgr_s = _extraer_biomarcador(texto, "PGR")
    mk_v, mk_s  = _extraer_biomarcador(texto, "MKI67")

    return {
        "nhc": None,
        "sample_id": sample_id,
        "fecha_informe": fecha,
        "subtipo_mmt": subtipo_mmt,
        "subtipo_mmt_detalle": subtipo_detalle,
        "ERBB2_value": erbb2_v,
        "ERBB2_status": erbb2_s,
        "ESR1_value": esr1_v,
        "ESR1_status": esr1_s,
        "PGR_value": pgr_v,
        "PGR_status": pgr_s,
        "MKI67_value": mk_v,
        "MKI67_status": mk_s,
        "texto_parcial": texto[:2000]
    }


def extraer_registros_pdf(file) -> List[Dict[str, Any]]:
    """Devuelve una lista de muestras, una por página del PDF."""
    reader = PdfReader(file)
    registros = []

    for page in reader.pages:
        txt = page.extract_text()
        if not txt:
            continue
        reg = _extraer_registro_pagina(txt)
        if reg:
            registros.append(reg)

    return registros


def extraer_desde_pdf(file) -> Dict[str, Any] | None:
    """Versión que devuelve solo la primera muestra (compatibilidad)."""
    regs = extraer_registros_pdf(file)
    return regs[0] if regs else None


# =========================
#  FUSIÓN EXCEL + PDF
# =========================

def fusionar_registro_patwin_pdf(
    reg_excel: Dict[str, Any],
    reg_pdf: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Fusiona la información de un registro del Excel y uno del PDF
    usando sample_id como referencia.
    """
    id_excel = reg_excel.get("sample_id")
    id_pdf = reg_pdf.get("sample_id")
    if id_excel and id_pdf and id_excel != id_pdf:
        aviso = f"WARNING: sample_id Excel={id_excel} != PDF={id_pdf}"
    else:
        aviso = None

    combinado: Dict[str, Any] = {}

    # Identificadores
    combinado["nhc"] = reg_excel.get("nhc") or reg_pdf.get("nhc")
    combinado["sample_id"] = id_pdf or id_excel

    # Info contexto (Excel)
    combinado["ronda"] = reg_excel.get("ronda")
    combinado["celularidad"] = reg_excel.get("celularidad")

    # Subtipos
    combinado["subtipo_ihq"] = reg_excel.get("subtipo_ihq")

    subtipo_mmt_pdf = reg_pdf.get("subtipo_mmt")
    subtipo_mmt_excel = reg_excel.get("subtipo_mmt")
    combinado["subtipo_mmt"] = subtipo_mmt_pdf or subtipo_mmt_excel
    combinado["subtipo_mmt_detalle"] = reg_pdf.get("subtipo_mmt_detalle")

    # Fecha informe
    combinado["fecha_informe_mmt"] = reg_pdf.get("fecha_informe")

    # Marcadores desde PDF
    for gen in ["ERBB2", "ESR1", "PGR", "MKI67"]:
        combinado[f"{gen}_value"] = reg_pdf.get(f"{gen}_value")
        combinado[f"{gen}_status"] = reg_pdf.get(f"{gen}_status")

    # Info IHQ desde Excel
    combinado["ERBB2_IHQ_SISH"] = reg_excel.get("ERBB2_IHQ_SISH")
    combinado["ESR1_IHQ"] = reg_excel.get("ESR1_IHQ")
    combinado["PGR_IHQ"] = reg_excel.get("PGR_IHQ")
    combinado["KI67_IHQ"] = reg_excel.get("KI67_IHQ")

    combinado["aviso"] = aviso

    return combinado


Overwriting extraccion.py


In [38]:
import extraccion

with open(r"C:\Users\Diego\Desktop\TFG\excel-anon.xlsx", "rb") as f:
    regs_excel = extraccion.extraer_registros_patwin(f)

len(regs_excel), regs_excel[0]


(8,
 {'nhc': None,
  'ronda': '4-fecha',
  'sample_id': 'DEMO_0',
  'celularidad': None,
  'subtipo_ihq': None,
  'subtipo_mmt': 'Luminal B-like (HER2 negative) — HR+ , HER2-',
  'ERBB2_value': 37.99,
  'ERBB2_status': 'Negative',
  'ERBB2_IHQ_SISH': None,
  'ESR1_value': 37.77,
  'ESR1_status': 'Positive',
  'ESR1_IHQ': None,
  'PGR_value': 35.78,
  'PGR_status': 'Positive',
  'PGR_IHQ': None,
  'MKI67_value': 33.96,
  'MKI67_status': 'Negative',
  'KI67_IHQ': None})

In [39]:
import importlib, extraccion
importlib.reload(extraccion)

with open(r"C:\Users\Diego\Desktop\TFG\mmt_uno.pdf", "rb") as f:
    reg_pdf = extraccion.extraer_desde_pdf(f)

reg_pdf


{'nhc': None,
 'sample_id': '1_25B11111B2',
 'fecha_informe': '07/11/25',
 'subtipo_mmt': 'Luminal B-like (HER2 negative)',
 'subtipo_mmt_detalle': 'HR+ , HER2 Low',
 'ERBB2_value': 39.98,
 'ERBB2_status': 'Negative',
 'ESR1_value': 41.6,
 'ESR1_status': 'Positive',
 'PGR_value': 35.33,
 'PGR_status': 'Positive',
 'MKI67_value': 34.64,
 'MKI67_status': 'Negative',
 'texto_parcial': 'Pathologist: Sample ID: 1_25B1 1111B2\nDepartment: Plate ID:\nDate of measurement: MammaTyper® Lot:\nDate of report: 07/11/25 Instrument: CFX96(Bio-Rad®)\n     Validity of Run\nWarning:Assay mix channel Assay NC PC\n1FAM ERBB2\nHEX ESR1\n2FAM MKI67\nHEX B2M\n3FAM PGR\nHEX CALM2\nSubtype According to St Gallen (2013) and\nASCO (2020)\nLuminal B-like (HER2 negative)\nHR+ , HER2 LowBiomarkerMammaTyper®\nResultsStatus\nERBB2 39.98 Negative\nESR1 41.6 Positive\nPGR 35.33 Positive\nMKI67 34.64 Negative\nAuthorized Signature  \n'}

In [33]:
import importlib, extraccion
importlib.reload(extraccion)


<module 'extraccion' from 'C:\\Users\\Diego\\Desktop\\TFG\\App\\extraccion.py'>

In [41]:
import extraccion, importlib
importlib.reload(extraccion)

# 1. Extraer registros del Excel
with open(r"C:\Users\Diego\Desktop\TFG\excel-anon.xlsx", "rb") as f:
    regs_excel = extraccion.extraer_registros_patwin(f)

reg_excel = regs_excel[0].copy()

# Simulamos que el Excel tiene el mismo sample_id que el PDF
reg_excel["sample_id"] = "1_25B11111B2"

# 2. Extraer registro del PDF
with open(r"C:\Users\Diego\Desktop\TFG\mmt_uno.pdf", "rb") as f:
    reg_pdf = extraccion.extraer_desde_pdf(f)

# 3. Fusionar
combinado = extraccion.fusionar_registro_patwin_pdf(reg_excel, reg_pdf)
combinado


{'nhc': None,
 'sample_id': '1_25B11111B2',
 'ronda': '4-fecha',
 'celularidad': None,
 'subtipo_ihq': None,
 'subtipo_mmt': 'Luminal B-like (HER2 negative)',
 'subtipo_mmt_detalle': 'HR+ , HER2 Low',
 'fecha_informe_mmt': '07/11/25',
 'ERBB2_value': 39.98,
 'ERBB2_status': 'Negative',
 'ESR1_value': 41.6,
 'ESR1_status': 'Positive',
 'PGR_value': 35.33,
 'PGR_status': 'Positive',
 'MKI67_value': 34.64,
 'MKI67_status': 'Negative',
 'ERBB2_IHQ_SISH': None,
 'ESR1_IHQ': None,
 'PGR_IHQ': None,
 'KI67_IHQ': None,
 'aviso': None}

In [46]:
import os

# borra el archivo de base de datos antiguo (si existe)
if os.path.exists("tfg_mamma.db"):
    os.remove("tfg_mamma.db")
    print("BD antigua borrada")
else:
    print("No existía tfg_mamma.db")

import importlib, db
importlib.reload(db)

# crea la BD nueva con el esquema que hemos definido
db.init_db()
print("BD nueva creada")


BD antigua borrada
BD nueva creada


In [47]:
db.insertar_muestra_combinada(combinado)
db.obtener_muestras()


[(1,
  None,
  '1_25B11111B2',
  '4-fecha',
  None,
  None,
  'Luminal B-like (HER2 negative)',
  'HR+ , HER2 Low',
  '07/11/25',
  39.98,
  'Negative',
  41.6,
  'Positive',
  35.33,
  'Positive',
  34.64,
  'Negative')]

In [3]:
import extraccion, importlib
importlib.reload(extraccion)
with open(r"C:\Users\Diego\Desktop\TFG\simulacro-patwin.xlsx", "rb") as f:
    regs = extraccion.extraer_registros_patwin(f)

len(regs), regs[0]


(9,
 {'nhc': None,
  'ronda': None,
  'sample_id': '25B11111',
  'celularidad': None,
  'subtipo_ihq': None,
  'subtipo_mmt': None,
  'ERBB2_value': None,
  'ERBB2_status': None,
  'ESR1_value': None,
  'ESR1_status': None,
  'PGR_value': None,
  'PGR_status': None,
  'MKI67_value': None,
  'MKI67_status': None,
  'ERBB2_IHQ_SISH': '4B5',
  'ESR1_IHQ': None,
  'PGR_IHQ': None,
  'KI67_IHQ': 20})

In [8]:
import pandas as pd

from extraccion import (
    extraer_registros_patwin,
    extraer_registros_pdf,
    fusionar_registro_patwin_pdf,
)

# 🔧 RUTAS (CÁMBIALAS A LO TUYO)
RUTA_EXCEL = r"C:\Users\Diego\Desktop\TFG\simulacro2_debug_envio.xlsx"   # o el Patwin real
RUTA_PDF   = r"C:\Users\Diego\Desktop\TFG\mmt_full.pdf"      # tu PDF MammaTyper
RUTA_SALIDA = r"C:\Users\Diego\Desktop\TFG\debug_extraccion_envio1.xlsx"


# 1) Extraer del EXCEL (Patwin)
with open(RUTA_EXCEL, "rb") as f:
    regs_patwin = extraer_registros_patwin(f)

df_patwin = pd.DataFrame(regs_patwin)


# 2) Extraer del PDF (MammaTyper)
with open(RUTA_PDF, "rb") as f:
    regs_pdf = extraer_registros_pdf(f)

df_pdf = pd.DataFrame(regs_pdf)


# 3) FUSIÓN por sample_id (solo para ver cómo quedarían en la app)
idx_excel = {
    r.get("sample_id"): r
    for r in regs_patwin
    if r.get("sample_id")
}

combinados = []
for reg_pdf in regs_pdf:
    sid = reg_pdf.get("sample_id")
    if not sid:
        continue
    reg_excel = idx_excel.get(sid)
    if reg_excel is None:
        continue
    combinado = fusionar_registro_patwin_pdf(reg_excel, reg_pdf)
    combinados.append(combinado)

df_fusion = pd.DataFrame(combinados)


# 4) Guardar todo en un único Excel con 3 hojas
with pd.ExcelWriter(RUTA_SALIDA, engine="openpyxl") as writer:
    df_patwin.to_excel(writer, sheet_name="patwin_extraccion", index=False)
    df_pdf.to_excel(writer, sheet_name="pdf_extraccion", index=False)
    df_fusion.to_excel(writer, sheet_name="fusion", index=False)

print(f"Archivo de debug guardado en:\n{RUTA_SALIDA}")


Archivo de debug guardado en:
C:\Users\Diego\Desktop\TFG\debug_extraccion_envio1.xlsx


In [7]:
from extraccion import extraer_registros_patwin

RUTA_EXCEL = r"C:\Users\Diego\Desktop\TFG\simulacro2_debug_envio.xlsx"

with open(RUTA_EXCEL, "rb") as f:
    regs_patwin = extraer_registros_patwin(f)

print("N registros:", len(regs_patwin))
print("Primeros 5:")
for r in regs_patwin[:5]:
    print(
        "sample_id:", r.get("sample_id"),
        "| ERBB2_IHQ_SISH:", r.get("ERBB2_IHQ_SISH"),
        "| ER_IHQ:", r.get("ESR1_IHQ"),
        "| PR_IHQ:", r.get("PGR_IHQ"),
        "| P53:", r.get("P53_IHQ_status"), r.get("P53_IHQ_pct"),
        "| Ki67:", r.get("KI67_IHQ"),
        "| CK19:", r.get("CK19_IHQ_status"),
    )


N registros: 52
Primeros 5:
sample_id: 25B11111 | ERBB2_IHQ_SISH: NEGATIVO (ULTRA LOW) | ER_IHQ: Positivo | PR_IHQ: Positivo | P53: Positivo 5 | Ki67: 20 | CK19: Positiva
sample_id: 25B13212 | ERBB2_IHQ_SISH: NEGATIVO (ULTRALOW) | ER_IHQ: None | PR_IHQ: None | P53: Mutado 90 | Ki67: 90 | CK19: Positiva
sample_id: 25B100009 | ERBB2_IHQ_SISH: negativo (score 0) | ER_IHQ: Positivo | PR_IHQ: Positivo | P53: Negativo 10 | Ki67: 10 | CK19: Positiva
sample_id: 25B12222 | ERBB2_IHQ_SISH: NEGATIVO (+; HER2 LOW). | ER_IHQ: Positivo | PR_IHQ: Positivo | P53: Negativo 3 | Ki67: 20 | CK19: Positiva
sample_id: 25B3333 | ERBB2_IHQ_SISH: equívoco (++) | ER_IHQ: Positivo | PR_IHQ: Positivo | P53: Mutado 15 | Ki67: 15 | CK19: Positiva


In [7]:
import importlib
import extraccion

importlib.reload(extraccion)

from extraccion import extraer_registros_patwin


In [8]:
import io
import pandas as pd
from extraccion import extraer_registros_patwin

# -----------------------------
# CONFIGURACIÓN: RUTA DEL EXCEL
# -----------------------------
ruta_excel = r"C:\Users\Diego\Desktop\TFG\simulacro2_debug_envio.xlsx"   # <-- CAMBIA ESTO

# -----------------------------
# CARGA DEL ARCHIVO
# -----------------------------
with open(ruta_excel, "rb") as f:
    excel_bytes = f.read()

excel_file = io.BytesIO(excel_bytes)

# -----------------------------
# EJECUCIÓN DEL PARSER
# -----------------------------
registros = extraer_registros_patwin(excel_file)

print(f"\n🔍 TOTAL REGISTROS EXTRAÍDOS: {len(registros)}\n")

# -----------------------------
# DEBUG DETALLADO
# -----------------------------
for i, reg in enumerate(registros, start=1):
    print("=" * 70)
    print(f"📌 REGISTRO {i}")
    print("=" * 70)

    for clave, valor in reg.items():
        print(f"{clave:35} → {valor}")

    print("\n")  # separación entre registros



🔍 TOTAL REGISTROS EXTRAÍDOS: 52

📌 REGISTRO 1
sample_id                           → 25B11111
ERBB2_IHQ_SISH                      → NEGATIVO (ULTRA LOW)
HER2_SISH_result                    → None
HER2_final                          → HER2 low (IHQ)
HER2_IHQ_score                      → None
ESR1_IHQ                            → Positivo
ESR1_IHQ_intensidad                 → +++/+++
PGR_IHQ                             → Positivo
PGR_IHQ_intensidad                  → +/+++
KI67_IHQ                            → 20
P53_IHQ_status                      → Positivo
P53_IHQ_pct                         → 5
CK19_IHQ_status                     → Positiva
pct_celulas_tumorales_positivas     → 100.0
firmantes_diag                      → None
ESR1_IHQ_pct                        → 100.0
PGR_IHQ_pct                         → 5.0


📌 REGISTRO 2
sample_id                           → 25B13212
ERBB2_IHQ_SISH                      → NEGATIVO (ULTRALOW)
HER2_SISH_result                    → None
HER2_final   